# GEAP Agent Evaluation — SDK-First (flat) Notebook

A **flat, teaching-oriented** walk of the same Quality Flywheel as
[`evaluation_demo.ipynb`](evaluation_demo.ipynb), but with the **Vertex AI GenAI evals SDK as a
first-class citizen** — every phase calls `client.evals.*` and `vertexai.types.*` **inline**, with no
`steps.*` wrapper. It covers Google's
[Optimize → Evaluation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/agent-evaluation) docs.

**L1 SDK vs custom.** We stick to the L1 SDK wherever possible. A few things are **not** eval-SDK
features and are called out with 🔧 — inference (a workaround for an SDK bug on the pinned version),
resilience/env-simulation, reading historical traces, online-monitor setup, the ADK GEPA optimizer,
and the Cloud Monitoring alert policy. Those reuse the repo's modules and are clearly labeled.

> Pinned to `google-cloud-aiplatform 1.162`. Where an L1 call is currently broken on that version the
> cell shows the documented call, the reason, and the minimal workaround.

## Setup

In [1]:
import os
# Run from the repo root so `from src...` imports and relative fixture paths resolve.
for _ in range(6):
    if os.path.exists("src/config.py"):
        break
    os.chdir("..")

# --- Vertex AI GenAI evals SDK: the L1 entry points used throughout ---
import vertexai
from vertexai import Client, types, agent_engines
from google.genai import types as g_types
from src.config import GCP_PROJECT_ID, GCP_REGION, AGENT_ENGINE_ID

vertexai.init(project=GCP_PROJECT_ID, location=GCP_REGION)
client = Client(project=GCP_PROJECT_ID, location=GCP_REGION)      # -> client.evals.* is the eval SDK

AGENT_RESOURCE = f"projects/{GCP_PROJECT_ID}/locations/{GCP_REGION}/reasoningEngines/{AGENT_ENGINE_ID}"
print("client.evals ready:", hasattr(client, "evals"), "| agent:", AGENT_RESOURCE)

/tmp/ipykernel_3165348/2323432565.py:15: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = Client(project=GCP_PROJECT_ID, location=GCP_REGION)      # -> client.evals.* is the eval SDK


23:59:56 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


23:59:56 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


client.evals ready: True | agent: projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024


### 🔧 One acknowledged helper: running inference

The documented L1 way to get responses is `client.evals.run_inference(agent=...)`. On
`aiplatform 1.162` it parses the Agent Engine's streamed events into the `AgentData` model
(`extra="forbid"`) and raises *"Failed to parse agent run response … to agent data: 'text'"*. So we
run inference ourselves via the Agent Engine `stream_query` API and hand the text to the SDK as
`EvalCase.responses`. **Everything else below is pure L1 SDK** (`client.evals.evaluate` + `types.*`).

In [2]:
# 🔧 CUSTOM (inference workaround) + tiny reporting helper. Scoring stays 100% L1 SDK.
_engine = agent_engines.get(AGENT_RESOURCE)

def agent_answer(prompt: str, user_id: str = "sdk-demo") -> str:
    """Query the deployed Agent Engine and return its final text (replaces run_inference on 1.162)."""
    texts = []
    for event in _engine.stream_query(message=prompt, user_id=user_id):
        for part in ((event.get("content") or {}).get("parts") or []):
            if part.get("text"):
                texts.append(part["text"])
    return "\n".join(texts).strip() or "(no response)"

def eval_case(prompt: str, answer: str, reference: str | None = None) -> types.EvalCase:
    """Wrap a prompt + response as an SDK EvalCase (extra='allow'; scored by client.evals.evaluate)."""
    kw = dict(
        prompt=g_types.Content(parts=[g_types.Part.from_text(text=prompt)], role="user"),
        responses=[types.ResponseCandidate(
            response=g_types.Content(parts=[g_types.Part.from_text(text=answer)], role="model"))],
    )
    if reference:
        kw["reference"] = g_types.Content(parts=[g_types.Part.from_text(text=reference)], role="model")
    return types.EvalCase(**kw)

def show_summary(result, threshold: float = 3.0):
    """🔧 reporting: SDK returns 0-1 scores; rescale x5 to the repo's 1-5 pass>=3 convention."""
    for m in (result.summary_metrics or []):
        mean = m.mean_score or 0.0
        score = mean * 5 if mean <= 1.0 else mean
        flag = "PASS" if score >= threshold else "FAIL"
        print(f"  {m.metric_name:34s} {score:4.2f}/5  [{flag}]  (errors={m.num_cases_error}/{m.num_cases_total})")

## Phase 1 — Design: the metric types (manage-metrics)
📖 [manage-metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics)

Three SDK metric types: **predefined rubric** (`types.RubricMetric.*`), **custom LLM-as-judge**
(`types.LLMMetric` + `types.MetricPromptBuilder`), and **custom deterministic code**
(`types.CodeExecutionMetric`). Define once, reuse — and optionally register in the Metric Registry
with `client.evals.create_evaluation_metric(...)`.

In [3]:
# 1) Predefined rubric metrics (Google-managed autoraters) — reference-free
prebuilt = [
    types.RubricMetric.FINAL_RESPONSE_QUALITY,
    types.RubricMetric.INSTRUCTION_FOLLOWING,
    types.RubricMetric.GENERAL_QUALITY,
]

# 2) Custom LLM-as-judge metric (natural-language rubric) — types.LLMMetric
policy_compliance = types.LLMMetric(
    name="policy_compliance",
    prompt_template=types.MetricPromptBuilder(
        instruction="Rate the agent's corporate expense-policy compliance.",
        criteria={"compliance": "Does the response correctly apply corporate expense limits and guide the user?"},
        rating_scores={"5": "proactive + correct", "4": "correct", "3": "applied, no guidance",
                       "2": "incorrect", "1": "ignores policy"},
    ),
)

# 3) Custom deterministic code metric — types.CodeExecutionMetric (server runs `evaluate(instance)->float`)
policy_limit_code = types.CodeExecutionMetric(
    name="policy_limit_exact",
    custom_function="""
def evaluate(instance: dict) -> float:
    text = str((instance or {}).get("response") or "").lower()
    limits = {"meal": 75, "meals": 75, "transport": 200, "lodging": 400, "supplies": 100, "entertainment": 150}
    hit = [c for c in limits if c in text]
    if not hit:
        return 0.5
    return 1.0 if any(str(limits[c]) in text for c in hit) else 0.0
""",
)

print("predefined rubric :", ["FINAL_RESPONSE_QUALITY", "INSTRUCTION_FOLLOWING", "GENERAL_QUALITY"])
print("custom LLM judge  :", policy_compliance.name, "(types.LLMMetric)")
print("custom code metric:", policy_limit_code.name, "(types.CodeExecutionMetric)")

# Register once, reuse across runs/monitors (writes to your project) — uncomment to run:
# print(client.evals.create_evaluation_metric(metric=policy_compliance))

predefined rubric : ['FINAL_RESPONSE_QUALITY', 'INSTRUCTION_FOLLOWING', 'GENERAL_QUALITY']
custom LLM judge  : policy_compliance (types.LLMMetric)
custom code metric: policy_limit_exact (types.CodeExecutionMetric)


## Phase 2a — Rapid evaluation (evaluate-agents)
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

Build an `EvaluationDataset`, score it with `client.evals.evaluate`, and render the SDK's interactive
table with `result.show()` (the [view-results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results) feature).

In [4]:
prompts = [
    "Find flights from SFO to JFK on June 15",
    "Search hotels in New York under $300",
    "Submit a $500 entertainment expense for user EMP001",
    "Check if a $50 meal expense is within policy",
    "Book flight FL001 for Jane Doe",
]
dataset = types.EvaluationDataset(eval_cases=[eval_case(p, agent_answer(p)) for p in prompts])

# --- L1 SDK scoring ---
result = client.evals.evaluate(dataset=dataset, metrics=prebuilt)
result.show()            # interactive aggregate + per-case tables
show_summary(result)

Computing Metrics for Evaluation Dataset:   0%|          | 0/15 [00:00<?, ?it/s]

Computing Metrics for Evaluation Dataset:   7%|▋         | 1/15 [00:15<03:39, 15.71s/it]

Computing Metrics for Evaluation Dataset:  13%|█▎        | 2/15 [00:16<01:27,  6.76s/it]

Computing Metrics for Evaluation Dataset:  20%|██        | 3/15 [00:17<00:52,  4.34s/it]

Computing Metrics for Evaluation Dataset:  27%|██▋       | 4/15 [00:18<00:32,  2.99s/it]

...

Computing Metrics for Evaluation Dataset:  80%|████████  | 12/15 [00:22<00:02,  1.47it/s]

Computing Metrics for Evaluation Dataset:  93%|█████████▎| 14/15 [00:23<00:00,  1.99it/s]

Computing Metrics for Evaluation Dataset: 100%|██████████| 15/15 [00:23<00:00,  1.88it/s]

Computing Metrics for Evaluation Dataset: 100%|██████████| 15/15 [00:23<00:00,  1.58s/it]

  final_response_quality_v1          2.67/5  [FAIL]  (errors=0/5)
  instruction_following_v1           4.03/5  [PASS]  (errors=0/5)
  general_quality_v1                 3.76/5  [PASS]  (errors=0/5)


## Phase 2b — Test-case / regression batch (evaluate-agents)
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

The same `client.evals.evaluate` call over a curated regression suite (a subset here for speed). The
test cases are project **data**; scoring is L1 SDK.

In [5]:
from src.eval.agent_eval_configs import get_eval_cases   # project test-case data (not SDK)

regression = get_eval_cases("coordinator_agent")[:6]
cases = [eval_case(c["prompt"], agent_answer(c["prompt"]), reference=c.get("reference")) for c in regression]
result = client.evals.evaluate(dataset=types.EvaluationDataset(eval_cases=cases), metrics=prebuilt)
show_summary(result)

Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


Type mismatch in EvalCase.reference: expected ResponseCandidate, got Content


...

Computing Metrics for Evaluation Dataset:  89%|████████▉ | 16/18 [00:32<00:05,  2.96s/it]

Computing Metrics for Evaluation Dataset:  94%|█████████▍| 17/18 [00:35<00:02,  2.84s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 18/18 [00:43<00:00,  4.29s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 18/18 [00:43<00:00,  2.40s/it]

  final_response_quality_v1          1.94/5  [FAIL]  (errors=0/6)
  instruction_following_v1           4.03/5  [PASS]  (errors=0/6)
  general_quality_v1                 2.88/5  [FAIL]  (errors=0/6)


## Phase 2c — Simulated scenario evaluation (evaluate-simulated)
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

`client.evals.generate_conversation_scenarios` synthesizes scenarios (a starting prompt + a hidden
conversation plan) grounded by an `environment_context`. We then run each scenario's opening turn and
score it. (Multi-turn `run_inference` is unavailable on 1.162, so we score the opening turn.)

In [6]:
from src.eval.agent_eval_configs import build_agent_info

agent_info = build_agent_info("coordinator_agent")

# --- L1 SDK: synthesize scenarios ---
scenarios = client.evals.generate_conversation_scenarios(
    agent_info=agent_info,
    config={
        "count": 2,
        "generation_instruction": "Generate diverse corporate travel + expense conversation scenarios.",
        "environment_context": "Employee EMP001. Meal limit $75, lodging $400. Flights FL001/FL002; hotels HT001/HT002.",
    },
    allow_cross_region_model=True,
)

cases = []
for sc in (scenarios.eval_cases or []):
    opening = (getattr(getattr(sc, "user_scenario", None), "starting_prompt", "") or "").strip()
    if opening:
        cases.append(eval_case(opening, agent_answer(opening)))

result = client.evals.evaluate(dataset=types.EvaluationDataset(eval_cases=cases), metrics=prebuilt)
show_summary(result)

Computing Metrics for Evaluation Dataset:   0%|          | 0/6 [00:00<?, ?it/s]

Computing Metrics for Evaluation Dataset:  17%|█▋        | 1/6 [00:16<01:23, 16.60s/it]

Computing Metrics for Evaluation Dataset:  33%|███▎      | 2/6 [00:16<00:28,  7.03s/it]

Computing Metrics for Evaluation Dataset:  50%|█████     | 3/6 [00:22<00:18,  6.26s/it]

Computing Metrics for Evaluation Dataset:  67%|██████▋   | 4/6 [00:22<00:07,  3.98s/it]

Computing Metrics for Evaluation Dataset:  83%|████████▎ | 5/6 [00:31<00:05,  5.73s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 6/6 [00:44<00:00,  8.29s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 6/6 [00:44<00:00,  7.48s/it]

  final_response_quality_v1          2.50/5  [FAIL]  (errors=0/2)
  instruction_following_v1           3.69/5  [PASS]  (errors=0/2)
  general_quality_v1                 3.67/5  [PASS]  (errors=0/2)


## Phase 2d — Environment simulation / resilience (evaluate-simulated) — 🔧 custom
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

Injecting mocked data + tool failures is **not** an eval-SDK feature — it wraps the agent's MCP tools.
Shown for completeness; the resulting traces would feed the multi-turn autoraters.

In [7]:
# 🔧 CUSTOM: mock tools + inject an HTTP 503 every 3rd call to test recovery.
from src.eval.env_simulation import run_with_env_simulation
run_with_env_simulation(inject_errors=True)

Environment simulation demo — agent_name='travel_agent'
In production, replace each MCP toolset on the agent with an
intercepted version, then run the simulated-user eval:
    live_tools = {t.name: t for t in agent.tools}         # MCP tools
    wrapped    = wrap_tools(live_tools, mocks=..., error_every=3)
    # ...attach `wrapped` to the agent, then:
    run_simulated_eval(agent_resource, agent_name=..., multi_turn=True)

Wrapped 3 tools: ['check_expense_policy', 'search_flights', 'search_hotels']
Error injection: every 3 call(s)
------------------------------------------------------------------------
  call 1: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 'FL002', 'route': 'SFO->JFK', 'price': 510, 'airline': 'Delta'}]
  call 2: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 'FL002', 'route': 'SFO->JFK', 'price': 510, 'airline'

{'agent_name': 'travel_agent',
 'agent_resource': None,
 'tools_wrapped': ['check_expense_policy', 'search_flights', 'search_hotels'],
 'total_calls': 6,
 'mock_hits': 5,
 'injected_errors': 1,
 'error_every': 3}

## Phase 2e — Offline evaluation over historical traces (evaluate-offline)
📖 [evaluate-offline](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-offline)

Score **already-recorded** prompt/response pairs — no new inference. In production you'd read gen_ai
OTel traces from BigQuery (🔧 custom); here we load a bundled fixture and score it with the L1 SDK.

In [8]:
import json, pathlib

traces = [json.loads(x) for x in pathlib.Path("src/eval/sample_traces.jsonl").read_text().splitlines() if x.strip()]
cases = [eval_case(t["prompt"], t["response"]) for t in traces]     # historical pairs -> EvalCases

# --- L1 SDK scoring of recorded traces ---
result = client.evals.evaluate(
    dataset=types.EvaluationDataset(eval_cases=cases),
    metrics=[types.RubricMetric.FINAL_RESPONSE_QUALITY,
             types.RubricMetric.HALLUCINATION,
             types.RubricMetric.SAFETY],
)
show_summary(result)

Computing Metrics for Evaluation Dataset:   0%|          | 0/24 [00:00<?, ?it/s]

Computing Metrics for Evaluation Dataset:   4%|▍         | 1/24 [00:03<01:23,  3.63s/it]

Computing Metrics for Evaluation Dataset:   8%|▊         | 2/24 [00:04<00:42,  1.95s/it]

Computing Metrics for Evaluation Dataset:  12%|█▎        | 3/24 [00:04<00:27,  1.29s/it]

Computing Metrics for Evaluation Dataset:  17%|█▋        | 4/24 [00:05<00:18,  1.05it/s]

...

Computing Metrics for Evaluation Dataset:  92%|█████████▏| 22/24 [00:32<00:06,  3.05s/it]

Computing Metrics for Evaluation Dataset:  96%|█████████▌| 23/24 [00:48<00:06,  6.66s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 24/24 [00:48<00:00,  4.90s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 24/24 [00:49<00:00,  2.04s/it]

  final_response_quality_v1          1.25/5  [FAIL]  (errors=0/8)
  hallucination_v1                   5.00/5  [PASS]  (errors=0/8)
  safety_v1                          4.38/5  [PASS]  (errors=0/8)


## Phase 3 — Continuous evaluation with Online Monitors (evaluate-online) — 🔧 custom
📖 [evaluate-online](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online)

Online Monitors asynchronously score **live production** traces on a ~10-minute loop and export scores
to Cloud Logging + Cloud Monitoring. They're created via the evaluation/Discovery Engine API (a setup
script), not `client.evals.evaluate`.

In [9]:
# 🔧 CUSTOM: create/verify online monitors via the setup script:
#     python -m src.eval.setup_online_evaluators create | verify
print("Online Monitors: async scoring of live traces (Query -> Evaluate -> Report),")
print("exported to Cloud Logging + Cloud Monitoring. See src/eval/setup_online_evaluators.py")

Online Monitors: async scoring of live traces (Query -> Evaluate -> Report),
exported to Cloud Logging + Cloud Monitoring. See src/eval/setup_online_evaluators.py


## Phase 4 — Optimize agent prompts (optimize-agent) — close the flywheel 🔁
📖 [optimize-agent](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/optimize-agent)

The documented L1 optimizer is `client.optimizer.optimize(targets=[...], benchmark=result, tests=dataset)`.
It isn't shipped in `aiplatform 1.162`, so we feature-detect it and otherwise point to the repo's ADK
GEPA fallback (🔧 custom), which re-tunes the coordinator's instruction against its eval set.

In [10]:
# --- L1 SDK (documented) ---
optimizer = getattr(client, "optimizer", None)
if optimizer is not None and callable(getattr(optimizer, "optimize", None)):
    opt = client.optimizer.optimize(targets=["system_prompt"], benchmark=result, tests=dataset)
    print("SDK optimizer result:", opt)
else:
    print("client.optimizer is not available in aiplatform 1.162 (the documented API is not shipped here).")
    print("🔧 Fallback — live GEPA optimization of the coordinator's instruction (requires reachable MCP):")
    print("   GEAP_RUN_GEPA=1 uv run python -m src.optimize.run_optimize src/agents/coordinator")
    print("   (or, from Python:  from src.eval.sdk_optimize import sdk_optimize;")
    print("                      sdk_optimize(client, run=True, max_metric_calls=30) )")

client.optimizer is not available in aiplatform 1.162 (the documented API is not shipped here).
🔧 Fallback — live GEPA optimization of the coordinator's instruction (requires reachable MCP):
   GEAP_RUN_GEPA=1 uv run python -m src.optimize.run_optimize src/agents/coordinator
   (or, from Python:  from src.eval.sdk_optimize import sdk_optimize;
                      sdk_optimize(client, run=True, max_metric_calls=30) )


## Quality-drift alerts (quality-alerts) — 🔧 custom
📖 [quality-alerts](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/quality-alerts)

Export a Cloud Monitoring alert policy on the online-evaluator score metric, then apply it with gcloud.

In [11]:
# 🔧 CUSTOM: write the alert policy YAML (apply with `gcloud monitoring policies create --policy-from-file=...`)
from src.eval.quality_alerts import export_policy_yaml
print("policy file:", export_policy_yaml("src/eval/policies/quality_drift_policy.yaml"))

✓ Alert policy YAML written: src/eval/policies/quality_drift_policy.yaml
  Metric: aiplatform.googleapis.com/online_evaluator/scores (evaluation_metric_name=task_success) < 0.8
  Apply: gcloud monitoring policies create --policy-from-file=src/eval/policies/quality_drift_policy.yaml
policy file: src/eval/policies/quality_drift_policy.yaml


## Recap — L1 SDK vs. acknowledged custom code

**Pure L1 SDK (`client.evals.*` / `vertexai.types.*`):**
- `client.evals.evaluate(dataset, metrics)` — rapid, regression, simulated, and offline scoring
- `client.evals.generate_conversation_scenarios(...)` — synthetic multi-turn scenarios
- `client.evals.create_evaluation_metric(...)` — Metric Registry
- metrics: `types.RubricMetric.*`, `types.LLMMetric` + `types.MetricPromptBuilder`, `types.CodeExecutionMetric`
- data: `types.EvaluationDataset`, `types.EvalCase`, `types.ResponseCandidate`, `types.evals.AgentInfo`
- `result.show()` — view-results tables; documented `client.optimizer.optimize(...)` (feature-detected)

**🔧 Acknowledged custom (not eval-SDK, or SDK-bug workarounds):**
- inference via `agent_engines.stream_query` (works around the `run_inference` AgentData bug on 1.162)
- resilience / environment simulation (MCP tool mocking + fault injection)
- reading historical OTel traces from BigQuery
- online-monitor setup (`setup_online_evaluators.py`)
- ADK GEPA optimizer (`run_optimize.py`) — the fallback until `client.optimizer` ships
- Cloud Monitoring alert policy (gcloud)

Headless orchestrator + JSON report:
`uv run python -m src.eval.demo.full_eval_demo --agent-id $AGENT_ENGINE_ID`